<a href="https://colab.research.google.com/github/LeslyDar/PortafolioBase1/blob/main/Mi_Copia_de_PSSB_I_Colab_U1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Programación de Sistemas de Base I - Inspección de la Estructura de un Compilador y Procesamiento de Lenguajes
**Asignatura:** Programación de Sistemas de Base I (8.º Semestre)  
**Unidad I:** Introducción a la Compilación  
**Tiempo estimado:** 3 horas (Asíncrono / Guiado)  
**Repositorio Base / Entrega:** Integración con Moodle / GitHub

---


## 1. Objetivos de Aprendizaje
1. Identificar de forma tangible la diferencia entre ejecución compilada e interpretada utilizando las herramientas nativas del entorno (`gcc`, `python3`, `dis`, `ast`).
2. Explorar y visualizar la fase de Análisis Léxico y Sintáctico mediante la inspección del Árbol de Sintaxis Abstracta (AST) y la generación de Bytecode.
3. Comprender el rol de la Tabla de Símbolos en el seguimiento de identificadores durante el proceso de traducción.

---


## 2. Conexión Teórica (NotebookLM)
**Antes de continuar:** Accede al espacio de **NotebookLM del Curso** y realiza las siguientes preguntas de verificación:

- *¿Cuáles son las fases de la etapa de Análisis (Front-End) de un compilador?*  
**RESPUESTA:**  
El proceso de compilación se divide tradicionalmente en dos etapas principales: la de **Análisis (Front-End)** y la de **Síntesis (Back-End)**. La etapa de **Análisis (Front-End)** se enfoca en entender la estructura y el significado del programa fuente, controlar que esté bien formado e independiente de la máquina de destino, y registrar los resultados de su análisis en una representación intermedia.  
Esta etapa consta típicamente de las siguientes fases secuenciales e interconectadas:  
**1. Análisis Léxico o Lexicográfico (Scanner o Analizador Morfológico)**
* Función: Es la primera fase del compilador. Su tarea consiste en leer el flujo de caracteres del programa fuente de izquierda a derecha y agruparlos en secuencias de caracteres con cohesión lógica conocidas como lexemas.
* Resultado: Para cada lexema identificado, el analizador léxico genera un componente léxico llamado token (representado habitualmente en el formato <nombre-token, valor-atributo>), el cual envía a la fase de análisis sintáctico.
* Tareas adicionales: Elimina comentarios, espacios en blanco consecutivos, tabuladores e introduce información como el número de línea para facilitar el posterior reporte de errores.  
**2. Análisis Sintáctico (Parser)**
* Función: Recibe el flujo de tokens generado por el análisis léxico y verifica que la secuencia de estos tokens respete las reglas gramaticales (la sintaxis) del lenguaje de programación que se está compilando.
* Resultado: Utilizando gramáticas libres de contexto, comprueba la estructura jerárquica del código y genera una representación intermedia que suele tomar la forma de un árbol sintáctico o árbol sintáctico abstracto (AST).
* Estructura de control: En la mayoría de los compiladores, el analizador sintáctico es quien guía todo el proceso (ejecutando una compilación dirigida por la sintaxis), de modo que solicita componentes léxicos al analizador léxico conforme los va necesitando para construir el árbol.  
**3. Análisis Semántico**
* Función: Toma el árbol sintáctico y la información de la tabla de símbolos para comprobar la consistencia del significado o semántica del programa de acuerdo con la especificación del lenguaje.
* Resultado: El producto de esta fase es el mismo árbol sintáctico enriquecido con anotaciones semánticas o atributos que preparan la generación de código.
* Tareas críticas: La tarea principal de esta fase es la verificación o chequeo de tipos, donde se controla que los operadores trabajen sobre operandos compatibles (por ejemplo, evitar que se sume un entero con un tipo incompatible o comprobar que el índice de una matriz sea un entero). También valida la previa declaración y ámbito de los identificadores.  
**4. Generación de Código Intermedio**
* Función: Aunque sirve como puente hacia la etapa de síntesis (Back-End), es la fase de salida que completa la labor del Front-End
. Traduce el árbol sintáctico anotado en una representación o lenguaje intermedio (como código de tres direcciones o cuádruplas) de una máquina abstracta o virtual.
* Beneficio: Esta fase es fundamental porque el código generado es independiente de la arquitectura física final. Esto facilita el retargeting o la portabilidad del compilador: si se quiere llevar el compilador a una nueva máquina, se puede reutilizar el mismo Front-End y únicamente rediseñar el Back-End.  
**Estructuras Auxiliares Comunes en el Front-End**  
Para que estas fases realicen su trabajo, se apoyan en dos sistemas transversales:
* La Tabla de Símbolos: Es una estructura de datos (como tablas hash o árboles binarios) que almacena el nombre de cada identificador (variable, constante, función, etc.) junto con información esencial sobre su tipo, ámbito y dirección. Tanto el analizador léxico como el sintáctico y el semántico insertan y consultan registros en esta tabla de manera constante.
* El Manejador de Errores: Se encarga de capturar fallas léxicas, sintácticas o semánticas cometidas por el programador e informar de ellas de manera inteligible para facilitar la depuración.

- *¿Qué diferencia a un Lexema de un Token?*  
**RESPUESTA:**  
En el análisis léxico de un compilador, token, lexema y patrón son términos estrechamente interconectados, pero representan distintos niveles de abstracción:  
**1. Lexema (La cadena concreta)**  
* Definición: Es la secuencia real de caracteres en el programa fuente que el analizador léxico identifica como una unidad con cohesión lógica (un "componente léxico").
* Nivel: Representa el texto físico y literal escrito por el programador en el código fuente.
* Ejemplos:
  * En la instrucción x = x + y - 3;, las secuencias "x", "=", "x", "+", "y", "-", "3" y ";" son lexemas individuales.
  * Nombres específicos de variables declarados por el programador (como comision o fijo) son lexemas.  
  
  **2. Token (La categoría abstracta)**
* Definición: Es la categoría gramatical o símbolo abstracto que representa a un tipo de unidad léxica. Es el elemento atómico que se envía como terminal al analizador sintáctico para que procese la estructura del programa.
* Estructura: Habitualmente, un token se genera como un par de la forma 〈nombre-token, valor-atributo〉.
  * El nombre del token influye en las decisiones estructurales del análisis sintáctico.
  * El valor del atributo (que suele ser un puntero a la tabla de símbolos o un valor constante directo) contiene información adicional necesaria para el análisis semántico y la generación de código.
* Ejemplos: El lexema "3.1416" se traduce al token numero (con su valor de atributo correspondiente). El lexema "velocidad" se traduce al token id.  
**La analogía de la fruta**  
Una manera muy clara de asimilar estas diferencias es a través de una analogía cotidiana que plantean los textos de estudio:
* El Token equivale al concepto abstracto general: Fruta.
* El Lexema equivale a la fruta física y concreta que tienes en la mano: una manzana, un plátano o una pera. Al analizador sintáctico le basta con saber que allí va una "Fruta", sin necesidad de evaluar de cuál se trata hasta fases posteriores.
* El Patrón equivale a la definición biológica o regla que determina qué califica como tal (por ejemplo, "estructura carnosa que rodea las semillas de una planta").  
**Resumen de diferencias de relación**  
* Relación de correspondencia: Mientras que para una palabra clave (como el token while) hay una relación exclusiva de uno a uno (el token while solo puede corresponder al lexema exacto "while"), para tokens genéricos como id o numero existe una relación de uno a muchos (el token id representa a un número infinito de lexemas posibles como x, resultado, suma, etc.).
* Flujo de procesamiento: El analizador léxico lee caracteres, los evalúa utilizando un patrón para aislar un lexema válido en el flujo de entrada, y luego retorna el token correspondiente para el analizador sintáctico.

- *¿Qué función cumple la Tabla de Símbolos durante la compilación?*  
**RESPUESTA:**  
Durante el proceso de compilación, la Tabla de Símbolos (también llamada tabla de nombres o de identificadores) funciona como una estructura de datos central o diccionario que registra los identificadores definidos por el usuario y reúne toda la información necesaria sobre ellos. Esta estructura se halla muy acoplada tanto a la etapa de análisis como a la de síntesis, permaneciendo en memoria principalmente en tiempo de compilación.
Sus funciones críticas se pueden agrupar en las siguientes tareas principales:  
**1. Registro de Identificadores y sus Atributos**  
La tabla de símbolos permite asociar a cada nombre detectado en el programa fuente un conjunto de atributos indispensables para las fases posteriores:
* Variables: Almacena su nombre, tipo de dato, tamaño y la dirección de memoria asignada para tiempo de ejecución.
* Arreglos (arrays): Registra el número de dimensiones, así como los límites inferior y superior de cada dimensión.
* Funciones y procedimientos: Guarda la cantidad de parámetros formales, sus tipos correspondientes, el método de paso de argumentos y el tipo del valor de retorno.
* Ámbitos (scopes): Registra el nivel léxico o de anidamiento donde el identificador fue declarado para controlar su visibilidad.  
**2. Soporte para el Análisis Semántico (Chequeos de Contexto)**  
El compilador utiliza la tabla de símbolos para realizar todas las comprobaciones que dependen del contexto y hacer cumplir las reglas del lenguaje:
* Verificación de declaraciones: Permite comprobar que cualquier identificador usado en una sentencia haya sido declarado previamente en el código fuente.
* Detección de redeclaraciones: Asegura que no se declare el mismo identificador de manera duplicada en un mismo nivel o bloque.
* Chequeo de tipos: Facilita la labor del analizador semántico al proveer los tipos de los operandos involucrados en una expresión, verificando que coincidan de manera compatible con los operadores.
* Consistencia de parámetros: Comprueba en las llamadas a subprogramas que los parámetros reales pasados correspondan exactamente en cantidad, orden y tipo con los parámetros formales de su declaración.  
**3. Generación de Código (Etapa de Síntesis)**  
Al final de la compilación, durante la generación de código intermedio o de máquina, la tabla de símbolos es consultada por el generador de código para sustituir los nombres simbólicos de las variables por las direcciones de memoria reales o desplazamientos de pila correspondientes en el código de destino.  
**4. Interacción con el Analizador Léxico (Scanner)**  
La tabla sirve para agilizar el trabajo del analizador léxico. Al preinicializarse con las palabras clave reservadas del lenguaje, permite distinguir si un texto leído corresponde a una variable del usuario o a una palabra del sistema (como if o while). También permite identificar nombres definidos como tipos nuevos por el usuario (por ejemplo, mediante typedef en C) para diferenciarlos de identificadores comunes.  
**5. Eficiencia en la Compilación**  
Al centralizar la información, evita la necesidad de buscar repetidamente a través del árbol de análisis sintáctico o de la representación intermedia para encontrar la declaración original de un identificador. Esto ahorra tiempo y optimiza la velocidad del compilador.


---

## 3. Sección Práctica 1: Pipeline de Compilación vs. Interpretación (CLI / Bash)
En esta sección analizaremos cómo el sistema operativo y el entorno procesan un lenguaje compilado (C) frente a uno interpretado/híbrido (Python).

### [CÓDIGO 1.1] Inspección de Herramientas del Sistema
```bash
!echo "=== COMPILADOR C DE LINUX (GCC) ==="
!gcc --version | head -n 1
!echo ""
!echo "=== INTÉRPRETE DE PYTHON ==="
!python3 --version

In [ ]:
!echo "=== COMPILADOR C DE LINUX (GCC) ==="
!gcc --version | head -n 1
!echo ""
!echo "=== INTÉRPRETE DE PYTHON ==="
!python3 --version

=== COMPILADOR C DE LINUX (GCC) ===
gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0

=== INTÉRPRETE DE PYTHON ===
Python 3.13.15


### [CÓDIGO 1.2] Creación, Compilación y Análisis de Binario en C
```c
%%writefile hola_compilador.c
#include <stdio.h>
#define inputMsj "Escriba un numero entero: "
#define msj "La suma de %d + %d es: %d\n"

int sumar(int x, int y){
  return x + y;
}

int main() {
    int a = 5;
    int b;
    
    printf(inputMsj);
    scanf("%d",&b);
    int suma = sumar(a, b);
    printf(msj, a, b, suma);
    return 0;
}

In [ ]:
%%writefile hola_compilador.c
# include <stdio.h>
# define inputMsj "Escriba un numero entero: "
# define msj "La suma de %d + %d es: %d\n"

int sumar(int x, int y){
  return x + y;
}

int main() {
    int a = 5;
    int b;

    printf(inputMsj);
    scanf("%d",&b);
    int suma = sumar(a, b);
    printf(msj, a, b, suma);
    return 0;
}

Writing hola_compilador.c


## El Preprocesador (Antes del ensamblador)
Esta etapa expande las macros (como `#define`) e incluye los archivos de cabecera (como `#include <stdio.h>`).
```bash
!gcc -E hola_compilador.c -o hola_compilador.i
!echo === CÓDIGO PREPROCESADO ===
!cat hola_compilador.i | tail -n 20


In [ ]:
!gcc -E hola_compilador.c -o hola_compilador.i
!echo "=== CÓDIGO PREPROCESADO ==="
!cat hola_compilador.i | tail -n 20

=== CÓDIGO PREPROCESADO ===
# 2 "hola_compilador.c" 2




# 5 "hola_compilador.c"
int sumar(int x, int y){
  return x + y;
}

int main() {
    int a = 5;
    int b;

    printf("Escriba un numero entero: ");
    scanf("%d",&b);
    int suma = sumar(a, b);
    printf("La suma de %d + %d es: %d\n", a, b, suma);
    return 0;
}


## Representación Intermedia (IR) y Optimizaciones
GCC utiliza una representación intermedia llamada GIMPLE antes de generar el ensamblador. Puedes ver cómo cambia el código antes y después de que el optimizador trabaje.

```bash
!gcc -O2 -fdump-tree-gimple hola_compilador.c -c
!echo === REPRESENTACIÓN INTERMEDIA (GIMPLE) ===
!cat hola_compilador.c.*gimple

In [ ]:
!gcc -O2 -fdump-tree-gimple hola_compilador.c -c
!echo "=== REPRESENTACIÓN INTERMEDIA (GIMPLE) ==="
!cat hola_compilador.c.*gimple

hola_compilador.c: In function ‘main’:
hola_compilador.c:14:5: warning: ignoring return value of ‘scanf’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]
   14 |     scanf("%d",&b);
      |     ^~~~~~~~~~~~~~
=== REPRESENTACIÓN INTERMEDIA (GIMPLE) ===
int sumar (int x, int y)
{
  int D.2555;

  D.2555 = x + y;
  return D.2555;
}


int main ()
{
  int D.2557;

  {
    int a;
    int b;
    int suma;

    try
      {
        a = 5;
        printf ("Escriba un numero entero: ");
        scanf ("%d", &b);
        b.0_1 = b;
        suma = sumar (a, b.0_1);
        b.1_2 = b;
        printf ("La suma de %d + %d es: %d\n", a, b.1_2, suma);
        D.2557 = 0;
        return D.2557;
      }
    finally
      {
        b = {CLOBBER};
      }
  }
  D.2557 = 0;
  return D.2557;
}


__attribute__((artificial, gnu_inline, always_inline))
__attribute__((nonnull (1), format (printf, 1, 2)))
int prin

## Compilación generando código ensamblador intermediate (.s)
 El compilador traduce el código **sin optimizaciones** línea por línea directamente a la memoria RAM (la pila o stack)

```bash
!gcc -S hola_compilador.c -o hola_compilador.s
!echo "=== CÓDIGO ENSAMBLADOR GENERADO (SÍNTESIS) ==="
!cat hola_compilador.s | head -n 25

In [ ]:
!gcc -S hola_compilador.c -o hola_compilador.s
!echo "=== CÓDIGO ENSAMBLADOR GENERADO (SÍNTESIS) ==="
!cat hola_compilador.s | head -n 25

=== CÓDIGO ENSAMBLADOR GENERADO (SÍNTESIS) ===
	.file	"hola_compilador.c"
	.text
	.globl	sumar
	.type	sumar, @function
sumar:
.LFB0:
	.cfi_startproc
	endbr64
	pushq	%rbp
	.cfi_def_cfa_offset 16
	.cfi_offset 6, -16
	movq	%rsp, %rbp
	.cfi_def_cfa_register 6
	movl	%edi, -4(%rbp)
	movl	%esi, -8(%rbp)
	movl	-4(%rbp), %edx
	movl	-8(%rbp), %eax
	addl	%edx, %eax
	popq	%rbp
	.cfi_def_cfa 7, 8
	ret
	.cfi_endproc
.LFE0:
	.size	sumar, .-sumar
	.section	.rodata


Para producir una traducción del código **optimizada** usa

```bash
!gcc -O3 -S hola_compilador.c -o hola_conmpilador_opt.s
!echo "=== CÓDIGO ENSAMBLADOR OPTIMIZADO GENERADO (SÍNTESIS) ==="
!cat hola_compilador_opt.s | head -n 25

In [ ]:
!gcc -O3 -S hola_compilador.c -o hola_compilador_opt.s
!echo "=== CÓDIGO ENSAMBLADOR OPTIMIZADO GENERADO (SÍNTESIS) ==="
!cat hola_compilador_opt.s | head -n 25

hola_compilador.c: In function ‘main’:
hola_compilador.c:14:5: warning: ignoring return value of ‘scanf’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]
   14 |     scanf("%d",&b);
      |     ^~~~~~~~~~~~~~
=== CÓDIGO ENSAMBLADOR OPTIMIZADO GENERADO (SÍNTESIS) ===
	.file	"hola_compilador.c"
	.text
	.p2align 4
	.globl	sumar
	.type	sumar, @function
sumar:
.LFB23:
	.cfi_startproc
	endbr64
	leal	(%rdi,%rsi), %eax
	ret
	.cfi_endproc
.LFE23:
	.size	sumar, .-sumar
	.section	.rodata.str1.1,"aMS",@progbits,1
.LC0:
	.string	"Escriba un numero entero: "
.LC1:
	.string	"%d"
.LC2:
	.string	"La suma de %d + %d es: %d\n"
	.section	.text.startup,"ax",@progbits
	.p2align 4
	.globl	main
	.type	main, @function


## Superando limitaciones de la arquitectura fija (X86_64)
La máquina virtual de Colab corre sobre procesadores Intel o AMD de 64 bits.  
**Impacto:** El código ensamblador generado (.s) siempre estará en sintaxis AT&T (por defecto en Linux) y para arquitectura x86_64. Para ver código ensamblador con sintaxis de Intel (más parecida a la de Windows), agregar el parámetro ***-masm=intel*** a la orden de GCC:

```bash
!gcc -S -masm=intel hola_compilador.c -o hola_compilador_8664.s
!echo "=== CÓDIGO ENSAMBLADOR x86_64 GENERADO (SÍNTESIS) ==="
!cat hola_compilador_8664.s | head -n 25


In [ ]:
!gcc -S -masm=intel hola_compilador.c -o hola_compilador_8664.s
!echo "=== CÓDIGO ENSAMBLADOR x86_64 GENERADO (SÍNTESIS) ==="
!cat hola_compilador_8664.s | head -n 25

=== CÓDIGO ENSAMBLADOR x86_64 GENERADO (SÍNTESIS) ===
	.file	"hola_compilador.c"
	.intel_syntax noprefix
	.text
	.globl	sumar
	.type	sumar, @function
sumar:
.LFB0:
	.cfi_startproc
	endbr64
	push	rbp
	.cfi_def_cfa_offset 16
	.cfi_offset 6, -16
	mov	rbp, rsp
	.cfi_def_cfa_register 6
	mov	DWORD PTR -4[rbp], edi
	mov	DWORD PTR -8[rbp], esi
	mov	edx, DWORD PTR -4[rbp]
	mov	eax, DWORD PTR -8[rbp]
	add	eax, edx
	pop	rbp
	.cfi_def_cfa 7, 8
	ret
	.cfi_endproc
.LFE0:
	.size	sumar, .-sumar


## Compilación a código objeto
```bash
# Generación del código objeto (antes de enlazar) y conteo líneas
!gcc -c hola_compilador.c -o hola_compilador.o
!objdump -d hola_compilador.o | wc -l

In [ ]:
# Generación del código objeto (antes de enlazar) y conteo líneas
!gcc -c hola_compilador.c -o hola_compilador.o
!objdump -d hola_compilador.o | wc -l

60


## Compilación a código máquina ejecutable

```bash
# Generación del código ejecutable final (después de enlazar) y conteo líneas
!gcc hola_compilador.c -o hola_compilador
!objdump -d hola_compilador | wc -l


In [ ]:
# Generación del código ejecutable final (después de enlazar) y conteo líneas
!gcc hola_compilador.c -o hola_compilador
!objdump -d hola_compilador | wc -l

194


## El "Monstruo Completo" (Enlace Estático)  
Por defecto, GCC usa ***enlace dinámico*** usando una estructura de datos tabular, la tabla **PLT** (*Procedure Linkage Table*) ya que el código usa `printf`, el programa necesita conectarse con la biblioteca estándar de C (`libc.so`). El enlazador no copia el código de `printf` dentro del archivo para no duplicar espacio en el disco; en su lugar, crea un "trampolín" o acceso directo en una sección llamada `<printf@plt>`.

Cada vez que se invoca a `printf`, realmente salta a este bloque plt, el cual averigua en qué parte de la memoria RAM del sistema operativo se encuentra la verdadera función `printf` y redirige el control allí en tiempo de ejecución.

Para ver cómo el enlazador inyecta literalmente miles y miles de líneas copiando físicamente el código de `printf` y todas sus dependencias dentro del ejecutable, se debe compilar de forma estática:

```bash
# Compilación estática
!gcc -static hola_compilador.c -o hola_estatico

# Conteo de líneas de código ensamblador
!objdump -d hola_estatico | wc -l


In [ ]:
# Compilación estática
!gcc -static hola_compilador.c -o hola_estatico

# Conteo de líneas de código ensamblador
!objdump -d hola_estatico | wc -l

182934


## Ejecución del código
```bash
!echo ""
!echo "=== EJECUCIÓN DEL PROGRAMA OBJETO ==="
!./hola_compilador


In [ ]:
!echo ""
!echo "=== EJECUCIÓN DEL PROGRAMA OBJETO ==="
!./hola_compilador


=== EJECUCIÓN DEL PROGRAMA OBJETO ===
Escriba un numero entero: 4
La suma de 5 + 4 es: 9


## 4. Sección Práctica 2: Revelando el Front-End (Análisis Léxico y AST)
Para comprender cómo el compilador "desarma" el código fuente en tokens y estructuras jerárquicas, inspeccionaremos el compilador de Python a nivel interno.

### [CÓDIGO 2.1] Inspección del Analizador Léxico (Tokenizador)

```python
import token
import tokenize
from io import BytesIO

# Código fuente de prueba en formato cadena
codigo_fuente = "suma = a + 10"

# Tokenización del código fuente
tokens = tokenize.tokenize(BytesIO(codigo_fuente.encode("utf-8")).readline)

print(
    f"{'LINEA/COL':<12} | {'TIPO DE TOKEN':<20} | {'VALOR (LEXEMA)':<15}"
)
print("-" * 55)
for tok in tokens:
    if tok.type in (
        tokenize.ENCODING,
        tokenize.ENDMARKER,
        tokenize.NL,
        tokenize.NEWLINE,
    ):
        continue
    nombre_token = token.tok_name[tok.type]
    posicion = f"{tok.start[0]}:{tok.start[1]}"
    print(f"{posicion:<12} | {nombre_token:<20} | {tok.string:<15}")


In [ ]:
import token
import tokenize
from io import BytesIO

# Código fuente de prueba en formato cadena
codigo_fuente = "suma = a + 10"

# Tokenización del código fuente
tokens = tokenize.tokenize(BytesIO(codigo_fuente.encode("utf-8")).readline)

print(
    f"{'LINEA/COL':<12} | {'TIPO DE TOKEN':<20} | {'VALOR (LEXEMA)':<15}"
)
print("-" * 55)
for tok in tokens:
    if tok.type in (
        tokenize.ENCODING,
        tokenize.ENDMARKER,
        tokenize.NL,
        tokenize.NEWLINE,
    ):
        continue
    nombre_token = token.tok_name[tok.type]
    posicion = f"{tok.start[0]}:{tok.start[1]}"
    print(f"{posicion:<12} | {nombre_token:<20} | {tok.string:<15}")

LINEA/COL    | TIPO DE TOKEN        | VALOR (LEXEMA) 
-------------------------------------------------------
1:0          | NAME                 | suma           
1:5          | OP                   | =              
1:7          | NAME                 | a              
1:9          | OP                   | +              
1:11         | NUMBER               | 10             


### [CÓDIGO 2.2] Construcción del Árbol de Sintaxis Abstracta (AST)

```python
import ast

# Generar y visualizar la estructura sintáctica
arbol = ast.parse("suma = a + 10")
print("=== ÁRBOLES DE SINTAXIS ABSTRACTA (REPRESENTACIÓN EN TEXTO) ===")
print(ast.dump(arbol, indent=4))


In [ ]:
import ast

# Generar y visualizar la estructura sintáctica
arbol = ast.parse("suma = a + 10")
print("=== ÁRBOLES DE SINTAXIS ABSTRACTA (REPRESENTACIÓN EN TEXTO) ===")
print(ast.dump(arbol, indent=4))

=== ÁRBOLES DE SINTAXIS ABSTRACTA (REPRESENTACIÓN EN TEXTO) ===
Module(
    body=[
        Assign(
            targets=[
                Name(id='suma', ctx=Store())],
            value=BinOp(
                left=Name(id='a', ctx=Load()),
                op=Add(),
                right=Constant(value=10)))])


## 5. Sección Práctica 3: Inspección de la Tabla de Símbolos y Bytecode
En esta sección simularemos la interacción con la Tabla de Símbolos y observaremos la generación de código intermedio.

### [CÓDIGO 3.1] Simulación de una Tabla de Símbolos básica en Python
```python
class TablaDeSimbolos:

    def __init__(self):
        self.simbolos = {}

    def insertar(self, nombre, tipo, valor=None, ambito="global"):
        if nombre in self.simbolos:
            print(f"[ERROR LÉXICO/SINTÁCTICO] Identificador '{nombre}' ya declarado.")
        else:
            self.simbolos[nombre] = {
                "tipo": tipo,
                "valor": valor,
                "ambito": ambito,
            }
            print(f"[TABLA DE SÍMBOLOS] Insertado: {nombre} ({tipo})")

    def buscar(self, nombre):
        return self.simbolos.get(nombre, None)

    def mostrar(self):
        print("\n=== CONTENIDO DE LA TABLA DE SÍMBOLOS ===")
        print(f"{'NOMBRE':<12} | {'TIPO':<10} | {'ÁMBITO':<10} | {'VALOR':<10}")
        print("-" * 50)
        for nombre, datos in self.simbolos.items():
            print(
                f"{nombre:<12} | {datos['tipo']:<10} | {datos['ambito']:<10} | {str(datos['valor']):<10}"
            )


# Prueba de la Tabla de Símbolos
ts = TablaDeSimbolos()
ts.insertar("a", "ENTERO", 5)
ts.insertar("b", "ENTERO", 10)
ts.insertar("suma", "ENTERO", 15)
ts.mostrar()
```


In [ ]:
class TablaDeSimbolos:

    def __init__(self):
        self.simbolos = {}

    def insertar(self, nombre, tipo, valor=None, ambito="global"):
        if nombre in self.simbolos:
            print(f"[ERROR LÉXICO/SINTÁCTICO] Identificador '{nombre}' ya declarado.")
        else:
            self.simbolos[nombre] = {
                "tipo": tipo,
                "valor": valor,
                "ambito": ambito,
            }
            print(f"[TABLA DE SÍMBOLOS] Insertado: {nombre} ({tipo})")

    def buscar(self, nombre):
        return self.simbolos.get(nombre, None)

    def mostrar(self):
        print("\n=== CONTENIDO DE LA TABLA DE SÍMBOLOS ===")
        print(f"{'NOMBRE':<12} | {'TIPO':<10} | {'ÁMBITO':<10} | {'VALOR':<10}")
        print("-" * 50)
        for nombre, datos in self.simbolos.items():
            print(
                f"{nombre:<12} | {datos['tipo']:<10} | {datos['ambito']:<10} | {str(datos['valor']):<10}"
            )


# Prueba de la Tabla de Símbolos
ts = TablaDeSimbolos()
ts.insertar("a", "ENTERO", 5)
ts.insertar("b", "ENTERO", 10)
ts.insertar("suma", "ENTERO", 15)
ts.mostrar()

[TABLA DE SÍMBOLOS] Insertado: a (ENTERO)
[TABLA DE SÍMBOLOS] Insertado: b (ENTERO)
[TABLA DE SÍMBOLOS] Insertado: suma (ENTERO)

=== CONTENIDO DE LA TABLA DE SÍMBOLOS ===
NOMBRE       | TIPO       | ÁMBITO     | VALOR     
--------------------------------------------------
a            | ENTERO     | global     | 5         
b            | ENTERO     | global     | 10        
suma         | ENTERO     | global     | 15        


### [CÓDIGO 3.2] Desensamblado a Código Intermedio / Bytecode (Máquina de Pila)
```python
import dis

def calcular():
    a = 5
    b = 10
    suma = a + b
    return suma


print("=== BYTECODE GENERADO PARA LA MÁQUINA VIRTUAL DE PYTHON ===")
dis.dis(calcular)
```

In [ ]:
import dis

def calcular():
    a = 5
    b = 10
    suma = a + b
    return suma


print("=== BYTECODE GENERADO PARA LA MÁQUINA VIRTUAL DE PYTHON ===")
dis.dis(calcular)

=== BYTECODE GENERADO PARA LA MÁQUINA VIRTUAL DE PYTHON ===
  3           RESUME                   0

  4           LOAD_CONST               1 (5)
              STORE_FAST               0 (a)

  5           LOAD_CONST               2 (10)
              STORE_FAST               1 (b)

  6           LOAD_FAST_LOAD_FAST      1 (a, b)
              BINARY_OP                0 (+)
              STORE_FAST               2 (suma)

  7           LOAD_FAST                2 (suma)
              RETURN_VALUE


## 6. Desafío Asíncrono / Entregable de la Unidad I (Avance del Producto Integrador)

### Contexto del Proyecto Integrador Autónomo:
Durante el semestre, cada equipo concebirá, diseñará e implementará un **Analizador Léxico y Sintáctico (Compiler Front-End)** para un lenguaje original propuesto por el propio equipo.

Ejemplos de proyectos de semestres anteriores:
- **DSL para Configuración de Robots / Drones:** Lenguaje de comandos simples (`FORWARD 10`, `ROTATE 90`).
- **Lenguaje de Consultas para Grafos/Tablas:** Alternativa simplificada a SQL (`SELECT age FROM users WHERE status == 1`).
- **Mini-Lenguaje Matemático / Scripting:** Soporte para vectores, matrices o evaluación de expresiones lógicas/aritméticas.
- **Lenguaje Formato/Marcado Personalizado:** Generador de reportes en HTML/Markdown a partir de una sintaxis limpia.

---

### Instrucciones del Desafío U1 (Fase 0: Definición e Inspección Inicial):

#### Parte A: Documento de Especificación del Lenguaje (RFC del Equipo)
Crea un archivo `ESPECIFICACION.docx` que contenga:
1. **Nombre del Lenguaje Original y Propósito:** ¿Qué problema resuelve o a quién va dirigido?
2. **Ejemplo de Código Fuente Válido:** Muestra un fragmento de código de al menos 10-15 líneas escrito en tu nuevo lenguaje.
3. **Catálogo Preliminar de Tokens:** Lista las palabras reservadas, identificadores, constantes (enteras, flotantes, cadenas) y operadores que usará tu lenguaje.

#### Parte B: Prototipo de Inspección en Código (Colab Execution)
Utilizando las celdas mágicas `%%writefile` (en Python o C):
1. Crea un archivo con una cadena que contenga tu ejemplo de código fuente original.
2. Implementa una función de prueba preliminar (o utiliza la librería `tokenize` de Python como simulador) que tome la cadena de tu lenguaje y la desglose en una lista de componentes léxicos.
3. Muestra una estructura en código que sirva como **Tabla de Símbolos inicial** donde se registren las variables declaradas en tu lenguaje.

---

### Criterios Mínimos que Debe Cumplir Cualquier Lenguaje Propuesto (Checklist de Viabilidad):
Para que la propuesta sea aprobada por el catedrático, el lenguaje debe cumplir con:
- [ ] Poseer al menos **3 tipos de tokens bien diferenciados** (ej. Palabras Reservadas, Identificadores, Literales).
- [ ] Incluir soporte para **expresiones aritméticas o lógicas** anidadas.
- [ ] Incluir al menos una **estructura de control de flujo** (ej. `if/else`, `while`, `repeat`) o una **estructura de bloques** (funciones, comandos).
- [ ] Definir un mecanismo explícito de asignación o declaración de datos.

---

### Autoevaluación Muestreada (Comprobación Tipo Gradiance)
Responde las siguientes preguntas analizando la sintaxis de tu nuevo lenguaje y valida tus razonamientos en NotebookLM:

1. **Pregunta 1 (Conflictos Léxicos):** Al revisar las palabras reservadas y los identificadores de tu lenguaje, ¿existe alguna regla léxica que pudiera causar ambigüedad (por ejemplo, que una palabra reservada coincida con el patrón de un identificador de usuario)? ¿Cómo la resolverá tu analizador?
2. **Pregunta 2 (Estructura de la Tabla de Símbolos):** De los elementos de tu lenguaje original, ¿cuáles atributos (tipo, valor, ámbito, dirección de memoria) necesitará almacenar tu Tabla de Símbolos cuando se procese una declaración?

---

### Formato de Entrega / Portafolio de Evidencias
1. Guarda este cuaderno con todas las salidas ejecutadas (`Archivo` -> `Guardar una copia en GitHub` / `PEREZ JUAN PSSB I Colab U1.ipynb`).
2. Guarda el documento `ESPECIFICACION.docx` con la especificación de tu lenguaje en tu repositorio personal (no repositorios grupales) de GitHub.
3. Registra en **Moodle** el enlace del cuaderno ejecutable en Colab y el de tu repositorio conteniendo la especificación formal del proyecto.

## Sección de trabajo no dirigido
Prototipo de Inspección en Código (Colab Execution)
Utilizando las celdas mágicas `%%writefile` (en Python o C):
1. Crea un archivo con una cadena que contenga tu ejemplo de código fuente original.

In [ ]:
%%writefile pipeline_test.pipespec
pipeline "deploy-service":
    stage "lint":
        image = "golang:1.24"
        exec:
            "golangci-lint run"

    stage "deploy" depends_on ["lint"]:
        image = "alpine/k8s:1.30"
        timeout = 180
        when = [[ branch == "main" ]]
        exec:
            "kubectl apply -f deploy/"

Writing pipeline_test.pipespec


2. Implementa una función de prueba preliminar (o utiliza la librería `tokenize` de Python como simulador) que tome la cadena de tu lenguaje y la desglose en una lista de componentes léxicos.

In [ ]:
import re

TOKENS_PIPESPEC = [
    ('KEYWORD',   r'\b(pipeline|stage|depends_on|when|exec)\b'),
    ('ID',        r'\b[a-zA-Z_][a-zA-Z0-9_\-]*\b'),
    ('NUMBER',    r'\b\d+\b'),
    ('STRING',    r'"[^"]*"'),
    ('OPERATOR',  r'==?'),
    ('COLON',     r':'),
    ('DELIM',     r'[\[\](),]'),
    ('SKIP',      r'[ \t\n]+'),
    ('MISMATCH',  r'.'),
]

def lexer_pipespec(codigo):
    patron_maestro = '|'.join(f'(?P<{nombre}>{patron})' for nombre, patron in TOKENS_PIPESPEC)
    tokens = []

    for match in re.finditer(patron_maestro, codigo):
        tipo = match.lastgroup
        valor = match.group(tipo)

        if tipo == 'SKIP':
            continue
        elif tipo == 'MISMATCH':
            raise SyntaxError(f"Error léxico: Carácter inesperado '{valor}'")
        else:
            tokens.append((tipo, valor))

    return tokens

with open("pipeline_test.pipespec", "r") as f:
    codigo_fuente = f.read()

tokens_resultado = lexer_pipespec(codigo_fuente)

print(f"{'TIPO DE TOKEN':<15} | {'VALOR (LEXEMA)'}")
print("-" * 35)
for t in tokens_resultado:
    print(f"{t[0]:<15} | {t[1]}")

TIPO DE TOKEN   | VALOR (LEXEMA)
-----------------------------------
KEYWORD         | pipeline
STRING          | "deploy-service"
COLON           | :
KEYWORD         | stage
STRING          | "lint"
COLON           | :
ID              | image
OPERATOR        | =
STRING          | "golang:1.24"
KEYWORD         | exec
COLON           | :
STRING          | "golangci-lint run"
KEYWORD         | stage
STRING          | "deploy"
KEYWORD         | depends_on
DELIM           | [
STRING          | "lint"
DELIM           | ]
COLON           | :
ID              | image
OPERATOR        | =
STRING          | "alpine/k8s:1.30"
ID              | timeout
OPERATOR        | =
NUMBER          | 180
KEYWORD         | when
OPERATOR        | =
DELIM           | [
DELIM           | [
ID              | branch
OPERATOR        | ==
STRING          | "main"
DELIM           | ]
DELIM           | ]
KEYWORD         | exec
COLON           | :
STRING          | "kubectl apply -f deploy/"


3. Muestra una estructura en código que sirva como **Tabla de Símbolos inicial** donde se registren las variables declaradas en tu lenguaje.

In [ ]:
class TablaSimbolosPipeSpec:
    def __init__(self):
        self.tabla = {}

    def declarar(self, nombre, tipo, valor, ambito="global"):
        self.tabla[nombre] = {
            "tipo": tipo,
            "valor": valor,
            "ambito": ambito
        }
        print(f"[TABLA DE SÍMBOLOS] Registrado: {nombre} ({tipo}) = {valor}")

    def mostrar(self):
        print("\n=== TABLA DE SÍMBOLOS (PIPESPEC) ===")
        print(f"{'NOMBRE':<15} | {'TIPO':<10} | {'VALOR':<25} | {'ÁMBITO':<10}")
        print("-" * 68)
        for k, v in self.tabla.items():
            print(f"{k:<15} | {v['tipo']:<10} | {str(v['valor']):<25} | {v['ambito']:<10}")

tsp = TablaSimbolosPipeSpec()
tsp.declarar("pipeline_name", "string", "\"deploy-service\"", "global")
tsp.declarar("stage_1", "string", "\"lint\"", "pipeline")
tsp.declarar("image_1", "string", "\"golang:1.24\"", "stage:lint")
tsp.declarar("stage_2", "string", "\"deploy\"", "pipeline")
tsp.declarar("timeout", "int", 180, "stage:deploy")
tsp.declarar("branch_condition", "string", "\"main\"", "stage:deploy")

tsp.mostrar()

[TABLA DE SÍMBOLOS] Registrado: pipeline_name (string) = "deploy-service"
[TABLA DE SÍMBOLOS] Registrado: stage_1 (string) = "lint"
[TABLA DE SÍMBOLOS] Registrado: image_1 (string) = "golang:1.24"
[TABLA DE SÍMBOLOS] Registrado: stage_2 (string) = "deploy"
[TABLA DE SÍMBOLOS] Registrado: timeout (int) = 180
[TABLA DE SÍMBOLOS] Registrado: branch_condition (string) = "main"

=== TABLA DE SÍMBOLOS (PIPESPEC) ===
NOMBRE          | TIPO       | VALOR                     | ÁMBITO    
--------------------------------------------------------------------
pipeline_name   | string     | "deploy-service"          | global    
stage_1         | string     | "lint"                    | pipeline  
image_1         | string     | "golang:1.24"             | stage:lint
stage_2         | string     | "deploy"                  | pipeline  
timeout         | int        | 180                       | stage:deploy
branch_condition | string     | "main"                    | stage:deploy


1. **Pregunta 1 (Conflictos Léxicos):** Al revisar las palabras reservadas y los identificadores de tu lenguaje, ¿existe alguna regla léxica que pudiera causar ambigüedad (por ejemplo, que una palabra reservada coincida con el patrón de un identificador de usuario)? ¿Cómo la resolverá tu analizador?

**RESPUESTA:**   
Sí puede haber un conflicto porque las palabras clave y los nombres que inventamos usan las mismas letras.

Cómo lo resolvemos?  
 Nuestro analizador léxico evita esto con dos reglas básicas de teoría de compiladores:

* Orden de prioridad: El sistema busca primero si la palabra es una palabra reservada del sistema antes de revisar si es un nombre común de usuario.

* Límites de palabras: Usamos una regla para asegurarnos de que la palabra reservada esté completa y no sea solo un pedazo dentro de otro nombre más largo.

2. **Pregunta 2 (Estructura de la Tabla de Símbolos):** De los elementos de tu lenguaje original, ¿cuáles atributos (tipo, valor, ámbito, dirección de memoria) necesitará almacenar tu Tabla de Símbolos cuando se procese una declaración?

**RESPUESTA:**   
Necesitará guardar los 4 atributos:

* Tipo: Para saber la naturaleza del dato y saber si una variable o parámetro es de tipo texto string o número entero int.

* Valor: Para registrar el dato exacto que se le asignó en el código fuente.

* Ámbito: Para saber en qué parte del código vive la variable, si pertenece al pipeline global o a una etapa específica, evitando que se mezclen nombres.

* Dirección de memoria: Para que el compilador tenga una referencia interna y sepa ubicar la posición exacta del elemento en la memoria.



